In [ ]:
# Block 1: notebook description and analysis objective

#This notebook is being used to evaluate momentum, efficiency, relative performance, and factor exposure for a single asset.
#Original Risk Analysis blocks included here: 15-21.


In [ ]:
# Block 2: load core libraries and instantiate helpers and plotters
# Load libraries
%load_ext autoreload
%autoreload 1
%aimport Quantapp.analytics
%aimport Quantapp.visualization
%aimport Quantapp.data
%aimport Quantapp.workflows

import logging
import warnings
import json
import os
import numpy as np
from scipy.stats import skew, kurtosis
from scipy.interpolate import PchipInterpolator
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
from plotly.subplots import make_subplots
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from Quantapp.visualization import Plotter, LineChartPlotter, CandleStickPlotter, BarChartPlotter, add_sigma_reference_lines, add_mean_reference_line, add_std_annotations, add_zone_annotation, add_horizontal_zone_trace, build_time_range_buttons, build_detail_visibility_mask, build_visibility_mask
from Quantapp.analytics import TimeSeriesAnalytics as Rolling, Helper, Models, MomentumAnalytics, RiskDistributionAnalytics, RiskRelativeAnalytics, SeriesTransforms, calculate_zscore, calculate_max_drawdown, gini_coefficient, calculate_window_metrics
from Quantapp.data import MacroDataClient, normalize_benchmark_tickers, load_benchmark_data, align_series_to_common_index
warnings.filterwarnings("ignore")
logger = logging.getLogger("yfinance")
logger.disabled = True
logger.propagate = False
rolling = Rolling()
series_transforms = SeriesTransforms()
momentum_analytics = MomentumAnalytics()
risk_relative_analytics = RiskRelativeAnalytics()
risk_distribution_analytics = RiskDistributionAnalytics()
qp = Plotter()
qe = MacroDataClient()
helper = Helper()
model = Models()
lineChartPlotter = LineChartPlotter()
candleStickPlotter = CandleStickPlotter()
barChartPlotter = BarChartPlotter()

PLOTLY_NOTEBOOK_CONFIG = {"responsive": True, "scrollZoom": True}
for renderer_name in ("plotly_mimetype", "notebook", "notebook_connected", "jupyterlab"):
    try:
        pio.renderers[renderer_name].config = PLOTLY_NOTEBOOK_CONFIG.copy()
    except Exception:
        pass

def show_plotly_figure(fig, *, config=None, **layout_kwargs):
    merged_config = PLOTLY_NOTEBOOK_CONFIG.copy()
    if config:
        merged_config.update(config)
    fig.update_layout(autosize=True, **layout_kwargs)
    fig.show(config=merged_config)


In [ ]:
# Block 3: configure analysis parameters
# Define parameters for the analysis (Adjust these as needed)
interval = '1d'
period     = '20y'
risk_free_ticker = '^IRX'  # Historical proxy used to derive the daily risk-free series
#should be a string or None
ticker_str ='XLF'#ticker to analyze
benchmark_tickers = ['SPY']  # Benchmarks to compare against (e.g., S&P 500, Bitcoin)
length_of_plots = 20 #number of years of data to plot (after computing the period, this will be adjusted to the closest available data)
trading_strategy = 'swing'  # Options: 'position', 'swing', or 'structural', # Determines the trading strategy to use for time frames
var_position_value = None  # Optional notional used to translate VaR / CVaR into dollar terms


In [ ]:
# Block 4: organize structural, position, swing timeframe lists

# Define strategy timeframe profiles
TIMEFRAME_PROFILES = {
    'swing': {'short': 3, 'mid': 9, 'long': 21},
    'position': {'short': 21, 'mid': 50, 'long': 200},
    'structural': {'short': 200, 'mid': 500, 'long': 1000},
}

strategy = str(trading_strategy).strip().lower()
if strategy not in TIMEFRAME_PROFILES:
    raise ValueError(
        f"Invalid trading_strategy '{trading_strategy}'. "
        f"Expected one of: {list(TIMEFRAME_PROFILES.keys())}"
    )

time_frame_map = TIMEFRAME_PROFILES[strategy]
time_frame_short = time_frame_map['short']
time_frame_mid = time_frame_map['mid']
time_frame_long = time_frame_map['long']

return_frequencies = ('monthly', 'weekly', 'daily')


In [ ]:
# Block 5: load and align asset, benchmark, and return data

# Download and normalize asset-level data
ticker = yf.Ticker(ticker_str).history(period=period, interval=interval)
vix = yf.Ticker('^VIX').history(period=period, interval=interval)
risk_free_proxy = yf.Ticker(risk_free_ticker).history(period=period, interval=interval)
ticker = helper.simplify_datetime_index(ticker)
vix = helper.simplify_datetime_index(vix)
risk_free_proxy = helper.simplify_datetime_index(risk_free_proxy)
if risk_free_proxy.empty or 'Close' not in risk_free_proxy:
    raise ValueError(f"No risk-free history available for {risk_free_ticker}.")
risk_free_annual_yield = risk_free_proxy['Close'].dropna().sort_index().div(100)
risk_free_daily_rate = ((1 + risk_free_annual_yield) ** (1 / 252) - 1).shift(1)

# Download benchmark data once and keep it in collections for downstream cells
benchmark_tickers = normalize_benchmark_tickers(benchmark_tickers, ticker_str)
benchmark_data, skipped_benchmarks = load_benchmark_data(benchmark_tickers, period, interval, helper)
if skipped_benchmarks:
    print(f'Skipped benchmarks with no data: {skipped_benchmarks}')

analysis_index, ticker, vix, benchmark_data = align_series_to_common_index(ticker, vix, benchmark_data)
risk_free_daily_rate = risk_free_daily_rate.reindex(ticker.index).ffill()

# Calculate asset and benchmark returns for the frequencies used elsewhere in the notebook
ticker_returns = {frequency: series_transforms.returns(ticker, frequency=frequency) for frequency in return_frequencies}
ticker_monthly_returns = ticker_returns['monthly']
ticker_weekly_returns = ticker_returns['weekly']
ticker_daily_returns = ticker_returns['daily']

benchmark_returns = {
    symbol: {frequency: series_transforms.returns(frame, frequency=frequency) for frequency in return_frequencies}
    for symbol, frame in benchmark_data.items()
}

vix_returns = {frequency: series_transforms.returns(vix, frequency=frequency) for frequency in return_frequencies}
vix_monthly_returns = vix_returns['monthly']
vix_weekly_returns = vix_returns['weekly']
vix_daily_returns = vix_returns['daily']


In [ ]:
# Block 6: build VIX Fix series and overlay standard deviation bands

#Volatility: VIX FIX 

ticker_vix_fix = rolling.vix_fix(ticker)
benchmark_vix_fix = {symbol: rolling.vix_fix(frame) for symbol, frame in benchmark_data.items()}

fig = qp.plot_series_with_stdev_bands(
    ticker_vix_fix,
    title='VIX Fix with Mean and Standard Deviations',
    stdev_values=[-0.5, 0.5, 1.5, 3],
    show=False,
)
show_plotly_figure(fig)


In [ ]:
# Block 7: stack candlestick, drawdown comparison, and rolling recovery time

peak_damage_window_options = [21, 50, 200]

peak_damage_context = risk_distribution_analytics.build_risk_distribution_context(
    close_series=ticker['Close'],
    windows=peak_damage_window_options,
    default_window=200 if 200 in peak_damage_window_options else max(peak_damage_window_options),
)

fig = lineChartPlotter.plot_candlestick_drawdown_recovery_profile(
    price_frame=ticker,
    metrics_by_window=peak_damage_context['metrics_by_window'],
    window_options=peak_damage_context['windows'],
    default_window=peak_damage_context['default_window'],
    ticker_label=ticker_str,
    candlestick_period=period,
    default_timeframe_label='10 Years',
)
show_plotly_figure(fig)


In [ ]:
# Block 8: render interactive momentum z-score comparisons

window_pairs = {
    "21 vs 50": (21, 50),
    "50 vs 200": (50, 200),
    "200 vs 400": (200, 400),
}

zscore_data = momentum_analytics.momentum_zscore_map(
    ticker['Close'],
    window_pairs=window_pairs,
)

fig = lineChartPlotter.plot_momentum_zscore_comparison(
    zscore_data=zscore_data,
    ticker_label=ticker_str,
    default_label="200 vs 400",
    default_time_label="3 Years",
    sigma_levels=(0.5, 1.0, 1.5),
)
fig.update_layout(height=850)
show_plotly_figure(fig)


In [ ]:
# Block 9: visualize monthly, weekly, and daily seasonality patterns

#Seasonality: Monthly, Weekly, and Daily Returns
#SEASONALITY: Monthly Seasonality
fig_ticker_Seasonality_Monthly = barChartPlotter.create_seasonality_fig(ticker_monthly_returns, f'Monthly Seasonality: {ticker_str}', 'monthly')
show_plotly_figure(fig_ticker_Seasonality_Monthly)

#SEASONALITY: Weekly Seasonality
fig_ticker_Seasonality_weekly = barChartPlotter.create_seasonality_fig(ticker_weekly_returns, f'Weekly Seasonality: {ticker_str}', 'weekly')
show_plotly_figure(fig_ticker_Seasonality_weekly)

#SEASONALITY: Daily Seasonality
fig_ticker_Seasonality_daily = barChartPlotter.create_seasonality_fig(ticker_daily_returns, f'Daily Seasonality: {ticker_str}', 'daily')
show_plotly_figure(fig_ticker_Seasonality_daily)

In [ ]:
# Block 10: compute Sharpe/Sortino ratios and spreads

import importlib
import Quantapp.analytics.risk_relative_analytics as risk_relative_analytics_module

# Refresh the analytics class in case the notebook kernel still holds an older signature.
importlib.reload(risk_relative_analytics_module)
risk_relative_analytics = risk_relative_analytics_module.RiskRelativeAnalytics()

asset_close = ticker['Close']

risk_context = risk_relative_analytics.build_sharpe_sortino_context(
    analytics=rolling,
    asset_close=asset_close,
    time_frame_map=time_frame_map,
    benchmark_data=benchmark_data,
    risk_free_rate=risk_free_daily_rate,
)

asset_sharpe_map = risk_context['asset_sharpe_map']
asset_component_map = risk_context['asset_component_map']
asset_sortino_map = risk_context['asset_sortino_map']
asset_sharpe_sortino_spread_map = risk_context['asset_sharpe_sortino_spread_map']

benchmark_metrics = risk_context['benchmark_metrics']
benchmark_order = risk_context['benchmark_order']
default_benchmark = risk_context['default_benchmark']
spread_plot_data = risk_context['spread_plot_data']
term_config_map = risk_context['term_config_map']


In [ ]:
# Block 11: plot Sharpe & Sortino efficiency with dropdown controls

long_window = time_frame_map.get('long')
default_term_label = f"{long_window}-day" if long_window is not None else max(term_config_map, key=lambda label: int(str(label).split('-')[0]))

fig = lineChartPlotter.plot_sharpe_sortino_comparison(
    term_config_map=term_config_map,
    ticker_label=ticker_str,
    default_label=default_term_label,
)
show_plotly_figure(fig)


In [ ]:
# Block 12: plot rolling correlation of the asset versus benchmarks

asset_daily_returns = ticker['Close'].pct_change(fill_method=None)
correlation_term_order = [term for term in time_frame_map if time_frame_map.get(term) is not None]
correlation_benchmark_order = benchmark_order if benchmark_order else list(benchmark_data.keys())
rolling_correlation_map = {}

for term in correlation_term_order:
    window = int(time_frame_map[term])
    term_series_map = {}

    for symbol in correlation_benchmark_order:
        benchmark_frame = benchmark_data.get(symbol)
        if benchmark_frame is None or 'Close' not in benchmark_frame:
            continue

        benchmark_daily_returns = benchmark_frame['Close'].pct_change(fill_method=None)
        aligned_returns = pd.concat(
            [
                asset_daily_returns.rename('asset'),
                benchmark_daily_returns.rename(symbol),
            ],
            axis=1,
        ).dropna()
        if aligned_returns.empty:
            continue

        rolling_correlation_series = aligned_returns['asset'].rolling(window).corr(aligned_returns[symbol]).dropna()
        if rolling_correlation_series.empty:
            continue

        term_series_map[symbol] = rolling_correlation_series

    if term_series_map:
        rolling_correlation_map[term] = term_series_map

if not rolling_correlation_map:
    print('No rolling correlation data available for benchmark comparison.')
else:
    rolling_correlation_palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
    benchmark_colors = {
        symbol: rolling_correlation_palette[idx % len(rolling_correlation_palette)]
        for idx, symbol in enumerate(correlation_benchmark_order)
    }
    correlation_rows = list(rolling_correlation_map.keys())
    rolling_correlation_fig = make_subplots(
        rows=len(correlation_rows),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=[f"{time_frame_map[term]}-Day Rolling Correlation" for term in correlation_rows],
    )

    for annotation in rolling_correlation_fig.layout.annotations:
        annotation.font = dict(size=11, color='rgba(220, 220, 220, 0.90)')

    date_ranges = []
    for row_idx, term in enumerate(correlation_rows, start=1):
        row_series_map = rolling_correlation_map[term]
        row_values = []

        for symbol_idx, (symbol, correlation_series) in enumerate(row_series_map.items()):
            row_values.append(correlation_series)
            date_ranges.append((correlation_series.index.min(), correlation_series.index.max()))
            rolling_correlation_fig.add_trace(
                go.Scatter(
                    x=correlation_series.index,
                    y=correlation_series,
                    mode='lines',
                    name=symbol,
                    legendgroup=symbol,
                    line=dict(color=benchmark_colors[symbol], width=2),
                    hovertemplate=f"Benchmark: {symbol}<br>Date: %{{x|%Y-%m-%d}}<br>Rolling Correlation: %{{y:.2f}}<extra></extra>",
                    showlegend=row_idx == 1,
                ),
                row=row_idx,
                col=1,
            )
            rolling_correlation_fig.add_hline(
                y=correlation_series.mean(),
                line_dash='dot',
                line_color=benchmark_colors[symbol],
                line_width=1,
                opacity=0.7,
                row=row_idx,
                col=1,
            )
            mean_annotation_x = max(0.18, 0.985 - (symbol_idx * 0.18))
            rolling_correlation_fig.add_annotation(
                x=mean_annotation_x,
                y=correlation_series.mean(),
                xref='x domain',
                yref='y',
                text=f"{symbol} Mean {correlation_series.mean():.2f}",
                showarrow=False,
                xanchor='right',
                yanchor='bottom',
                font=dict(size=10, color=benchmark_colors[symbol]),
                row=row_idx,
                col=1,
            )

        rolling_correlation_fig.add_hline(
            y=0,
            line_dash='dash',
            line_color='rgba(180, 180, 180, 0.65)',
            line_width=1,
            opacity=0.8,
            row=row_idx,
            col=1,
        )
        rolling_correlation_fig.add_annotation(
            x=0.985,
            y=0,
            xref='x domain',
            yref='y',
            text='Zero',
            showarrow=False,
            xanchor='right',
            yanchor='top',
            font=dict(size=10, color='rgba(200, 200, 200, 0.85)'),
            row=row_idx,
            col=1,
        )

        if row_values:
            row_frame = pd.concat(row_values, axis=1)
            row_min = min(row_frame.min().min(), 0)
            row_max = max(row_frame.max().max(), 0)
            row_span = row_max - row_min
            row_padding = max(row_span * 0.08, 0.03) if pd.notna(row_span) else 0.03
            rolling_correlation_fig.update_yaxes(
                title_text='Correlation',
                range=[row_min - row_padding, row_max + row_padding],
                row=row_idx,
                col=1,
            )

    if date_ranges:
        global_start = min(start for start, _ in date_ranges)
        global_end = max(end for _, end in date_ranges)
        default_start = max(global_start, global_end - pd.DateOffset(years=3))
        for axis_idx in range(1, len(correlation_rows) + 1):
            axis_name = 'xaxis' if axis_idx == 1 else f'xaxis{axis_idx}'
            rolling_correlation_fig.layout[axis_name].update(range=[default_start, global_end])
        time_range_menu = dict(
            buttons=build_time_range_buttons(global_start, global_end, axis_count=len(correlation_rows)),
            direction='down',
            showactive=True,
            x=0.01,
            y=1.12,
            xanchor='left',
            yanchor='top',
            active=2,
        )
    else:
        time_range_menu = None

    rolling_correlation_fig.update_layout(
        title=f'{ticker_str} Rolling Correlation vs Benchmarks',
        template='plotly_dark',
        height=max(760, 240 * len(correlation_rows) + 140),
        margin=dict(l=50, r=40, t=90, b=40),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
        updatemenus=[time_range_menu] if time_range_menu is not None else [],
    )
    rolling_correlation_fig.update_xaxes(title_text='Date', row=len(correlation_rows), col=1)
    show_plotly_figure(rolling_correlation_fig)


In [ ]:
# Block 13: combine risk-adjusted return and benchmark plots
# Requires the current kernel session to have fresh outputs from Blocks 2, 5, and 10.

if benchmark_order:
    benchmark_plot_payload = risk_relative_analytics.build_benchmark_plot_payload(
        asset_sharpe_map=asset_sharpe_map,
        asset_component_map=asset_component_map,
        benchmark_metrics=benchmark_metrics,
        spread_plot_data=spread_plot_data,
        time_frame_map=time_frame_map,
    )

    default_term = 'long' if 'long' in time_frame_map else max(time_frame_map, key=time_frame_map.get)

    summary_fig = lineChartPlotter.plot_multi_benchmark_sharpe_spread_summary(
        summary_zscore_map=benchmark_plot_payload['summary_zscore_map'],
        time_frame_map=time_frame_map,
        ticker_label=ticker_str,
        default_term=default_term,
    )
    show_plotly_figure(summary_fig)

    detail_fig = lineChartPlotter.plot_benchmark_zscore_detail(
        detail_zscore_map=benchmark_plot_payload['detail_zscore_map'],
        benchmark_order=benchmark_plot_payload['benchmark_order'],
        time_frame_map=time_frame_map,
        ticker_label=ticker_str,
        default_benchmark=benchmark_plot_payload['default_benchmark'],
        default_term=default_term,
    )
    show_plotly_figure(detail_fig)
else:
    print('No benchmark data available for benchmark comparison plots.')


In [ ]:
# Block 14: idiosyncratic risk via Fama-French Factor Analysis
analysis_period = "max"
interval = "1d"
rolling_window = 252

ff_proxy = model.run_ff5_proxy_analysis(
    asset_ticker=ticker_str,
    period=analysis_period,
    interval=interval,
    window=rolling_window,
    auto_window=True,
    verbose=True,
)

prices_df = ff_proxy["proxy_prices"]
returns = ff_proxy["proxy_returns"]
factor_returns = ff_proxy["factor_returns_all"]
factor_returns_capm = ff_proxy["factor_returns_capm"]
factor_returns_ff3 = ff_proxy["factor_returns_ff3"]
factor_returns_ff5 = ff_proxy["factor_returns_ff5"]
factor_returns_ff5_custom = factor_returns_ff5.copy()
stock_returns = ff_proxy["stock_returns"]
rolling_results_ff5_custom = ff_proxy["rolling_results"]

#Plotting the Fama-French Factor Analysis Results
qp.plot_rolling_regression(rolling_results_ff5_custom, ticker_str, factor_returns_ff5_custom)
idio_fig = qp.plot_idiosyncratic_risk(rolling_results_ff5_custom, ticker_str)
